# BnMMLU — full-dataset evaluation of six local models (v3)
Updated 21 September 2026. The supplied v2.2 results show all four previous models successfully completed 164-question smoke runs. This notebook replaces sampling with an **audit of every row in the `full` configuration** and schedules Qwen 1.5B, SmolLM2 1.7B, TituLLM 3B, TigerLLM 1B, Meta Llama 3.2 3B Instruct, and local Mixtral-8x7B-Instruct-v0.1.

**No hosted inference, generation API, Gemini, or API fallback is included.** Hugging Face is used to download datasets, tokenizers and weights; predictions are computed locally on Kaggle.

1. Enable Kaggle Internet and an available GPU. Accept the [BnMMLU](https://huggingface.co/datasets/samanjoy2/BnMMLU) and [Llama](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) access terms and enable the read-only `HF_TOKEN` secret.
2. Run setup first. Follow a kernel-restart message if one appears, then rerun from the top. The setup preserves Kaggle's CUDA PyTorch.
3. Leave `RESUME_DB = ""` for the first v3 run. v2 checkpoints are incompatible because the row population and calibration protocol changed. All six models are selected by default, one at a time.
4. Save/download the v3 output and attach it privately on the next session. Set `RESUME_DB` to its `checkpoint_export.sqlite` (or the intact live `checkpoint.sqlite`). All completed records are skipped automatically. You may select a subset of models per session without changing the protocol.
5. Expect multiple sessions for the entire dataset. The session budget is a soft limit checked between examples, not an interruption of an active model download/forward pass.

**Mixtral resource requirement:** the official checkpoint contains roughly 47B parameters, approximately 94 GB of BF16 weights before on-the-fly 4-bit quantization. A single 16 GB GPU is insufficient for this loader. Two 16 GB GPUs may fit short-context NF4 inference, subject to actual free memory. This notebook checks live memory and exact download sizes before loading and records a resource block rather than inventing results. Weights are cached under `/tmp` by default so they do not become a giant Kaggle output artifact. That filesystem still needs enough free space. NF4 changes precision and must be reported as a separate condition. [Official Mixtral model card](https://huggingface.co/mistralai/Mixtral-8x7B-Instruct-v0.1)

**Meaning of full dataset:** there is no per-subject sample, development-pool filter or deduplication. Every source row appears in `source_manifest.csv`. Structurally invalid rows cannot be scored and are counted explicitly. Context limits and execution failures are recorded per model, never silently truncated or converted into incorrect answers. Raw metrics describe all actually scored rows; coverage is always shown. Only `all_source_rows_scored=true` means literally every source row received a valid score for that model.

Calibration uses five-fold **question-grouped cross-fitting**: duplicate question stems always stay in the same fold, and a row's calibrated prediction uses a temperature fitted on other folds. This yields out-of-fold calibrated predictions across the scored dataset. Because the earlier smoke study already inspected some questions, this full-dataset extension is descriptive/exploratory, not an untouched confirmatory test. No confidence intervals, p-values, causal claims or publication-readiness claims are manufactured.

In [1]:
# Standard-library imports only until installation is complete.
import os
import sys
import subprocess
import importlib.metadata as setup_metadata
from pathlib import Path

if globals().get("_BNMMLU_RESTART_REQUIRED", False):
    raise RuntimeError("Restart the Python kernel, then run from the top. Rerunning this cell is not a restart.")
_BNMMLU_SETUP_READY = False
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# Compatible with the conflicts in the supplied Kaggle log. Preserve CUDA PyTorch.
# datasets 4.1.1 accepts dill 0.3.9 and fsspec 2025.3.0; 3.5.0 did not.
packages = [
    "transformers==4.56.2", "huggingface_hub==0.34.4",
    "datasets==4.1.1", "accelerate==1.10.1", "sentencepiece==0.2.0",
    "protobuf==5.29.5", "dill==0.3.9", "fsspec==2025.3.0",
    "statsmodels==0.14.5", "scipy==1.15.2",
    "regex==2024.11.6", "requests==2.32.4",
]
# Required for the selected local Mixtral NF4 model.
INSTALL_QUANTIZATION = True
if INSTALL_QUANTIZATION:
    packages.append("bitsandbytes==0.47.0")

def installed_version(name):
    try:
        return setup_metadata.version(name)
    except setup_metadata.PackageNotFoundError:
        return None

def environment_versions():
    return {d.metadata["Name"].lower().replace("_", "-"): d.version
            for d in setup_metadata.distributions() if d.metadata["Name"]}

before = environment_versions()
loaded_roots = {name.split(".")[0] for name in sys.modules}
loaded_distributions = {
    name.lower().replace("_", "-")
    for root, names in setup_metadata.packages_distributions().items()
    if root in loaded_roots for name in names
}
needs_install = any(installed_version(p.split("==")[0]) != p.split("==")[1] for p in packages)
if needs_install:
    # Keep the flag set even if pip fails halfway through an installation.
    _BNMMLU_RESTART_REQUIRED = True
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *packages])

# Text-only evaluation: optional media extensions can be ABI-incompatible with Kaggle torch.
media = [p for p in ("torchvision", "torchaudio", "librosa") if installed_version(p)]
if media:
    _BNMMLU_RESTART_REQUIRED = True
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", *media])

scipy_probe = "from scipy.special import softmax, logsumexp, expit, logit; from scipy.optimize import minimize_scalar; from scipy.stats import binomtest"
probe = subprocess.run([sys.executable, "-c", scipy_probe], capture_output=True, text=True)
repaired_scipy = probe.returncode != 0
if repaired_scipy:
    print("Repairing SciPy: its imports also fail in a fresh Python process.")
    _BNMMLU_RESTART_REQUIRED = True
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir",
                           "--force-reinstall", "--no-deps", "scipy==1.15.2"])
    subprocess.check_call([sys.executable, "-c", scipy_probe])

after = environment_versions()
changed = {name for name in before.keys() | after.keys() if before.get(name) != after.get(name)}
stale = changed & loaded_distributions
if repaired_scipy and "scipy" in loaded_roots:
    stale.add("scipy")
if stale:
    _BNMMLU_RESTART_REQUIRED = True
    raise RuntimeError("Installed packages changed after import: " + ", ".join(sorted(stale))
                       + ". Restart the Python kernel, then run from the top.")
_BNMMLU_RESTART_REQUIRED = False

# Check the actual text/scientific stack before any gated downloads or GPU allocation.
stack_probe = scipy_probe + "; import numpy, pandas, statsmodels.api; from datasets import load_dataset; from transformers import AutoModelForCausalLM, AutoTokenizer; import torch; print('Fresh-process scientific/model imports passed; torch', torch.__version__)"
subprocess.check_call([sys.executable, "-c", stack_probe])
try:
    from scipy.special import softmax, logsumexp, expit, logit
except ImportError:
    _BNMMLU_RESTART_REQUIRED = True
    raise RuntimeError("SciPy works in a fresh process but this kernel has stale modules. Restart Python and run from the top.") from None

# Check required dependency metadata recursively, ignoring optional extras.
from packaging.requirements import Requirement
todo = [p.split("==")[0] for p in packages] + ["numpy", "pandas", "matplotlib", "torch"]
seen, dependency_errors = set(), []
while todo:
    name = todo.pop()
    canonical = name.lower().replace("_", "-")
    if canonical in seen:
        continue
    seen.add(canonical)
    for text in setup_metadata.requires(name) or []:
        req = Requirement(text)
        if req.marker and not req.marker.evaluate({"extra": ""}):
            continue
        version = installed_version(req.name)
        if version is None or (req.specifier and not req.specifier.contains(version, prereleases=True)):
            dependency_errors.append(f"{name} requires {req}; installed={version}")
        else:
            todo.append(req.name)
if dependency_errors:
    raise RuntimeError("Required notebook dependency conflicts:\n" + "\n".join(dependency_errors))
_BNMMLU_SETUP_READY = True
print("Setup passed. Continue below. Unrelated Kaggle packages are not used by this study.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 240.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 337.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 391.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 358.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 335.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 330.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 374.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Configuration
All six local models are selected. To spread work across sessions, change only `RUN_MODELS` and the session time budget. Keep the panel, seed, precision, context policy and software environment fixed when resuming.

In [2]:
if not globals().get("_BNMMLU_SETUP_READY", False):
    raise RuntimeError(
        "Run setup first and follow any kernel-restart instruction."
    )

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass

import os
import json
import hashlib
import sqlite3
import time
import gc
import random
import shutil
import platform
import inspect
import warnings
import re
import unicodedata
import importlib.metadata as metadata

import numpy as np
import pandas as pd
import regex
import torch
import matplotlib.pyplot as plt

from scipy.special import softmax, logsumexp
from scipy.optimize import minimize_scalar
from huggingface_hub import HfApi
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from IPython.display import display, FileLink


# Dataset and reproducibility settings
DATASET = "samanjoy2/BnMMLU"
DATASET_CONFIG = "full"
DATASET_REVISION = "main"
SEED = 20260916

# Keep the complete panel so reports include all six models.
PANEL = [
    "qwen_1_5b",
    "smol_1_7b",
    "titu_3b",
    "tiger_1b",
    "llama3_3b",
    "mixtral_8x7b",
]

# Only these models will process pending questions in this session.
RUN_MODELS = ["llama3_3b", "tiger_1b", "mixtral_8x7b"]

# Soft limit checked between questions.
SESSION_MINUTES = 540

OUT = Path("/kaggle/working/bnmmlu_v3_full")
CACHE_ROOT = Path("/tmp/bnmmlu_v3_model_cache")
DB = OUT / "checkpoint.sqlite"

# Locate the previous checkpoint in the attached Kaggle dataset.
resume_folder = Path(
    "/kaggle/input/datasets/dipta2002/banglallm-dataset"
)
RESUME_DB = ""

if DB.is_file():
    print("Continuing from existing working checkpoint:", DB)
else:
    if not resume_folder.is_dir():
        raise FileNotFoundError(
            f"Input folder does not exist: {resume_folder}\n"
            "Attach your saved output and check its Kaggle input path."
        )

    checkpoints = sorted(
        resume_folder.rglob("checkpoint_export.sqlite")
    )

    if len(checkpoints) != 1:
        raise FileNotFoundError(
            "Expected exactly one checkpoint_export.sqlite in "
            f"{resume_folder}, found {len(checkpoints)}.\n"
            f"Matching paths: {[str(p) for p in checkpoints]}\n"
            "Attach the saved SQLite checkpoint, not only the CSV files."
        )

    RESUME_DB = str(checkpoints[0])
    print("Checkpoint selected for resume:", RESUME_DB)


RELEASE_COMPLETED_WEIGHTS = True
RETRY_FAILED_ROWS = True
MAX_CONSECUTIVE_ERRORS = 3
CALIBRATION_FOLDS = 5

# Preserve the original evaluation settings when resuming.
CONTEXT_CAP = None
ROTATIONS = [0]
BOOTSTRAPS = 0


@dataclass(frozen=True)
class ModelSpec:
    repo: str
    precision: str
    min_free_gpu_gib: float


REGISTRY = {
    "qwen_1_5b": ModelSpec(
        "Qwen/Qwen2.5-1.5B-Instruct", "fp16", 6
    ),
    "smol_1_7b": ModelSpec(
        "HuggingFaceTB/SmolLM2-1.7B-Instruct", "fp16", 6
    ),
    "titu_3b": ModelSpec(
        "hishab/titulm-llama-3.2-3b-v2.0", "fp16", 10
    ),
    "tiger_1b": ModelSpec(
        "md-nishat-008/TigerLLM-1B-it", "fp32", 8
    ),
    "llama3_3b": ModelSpec(
        "meta-llama/Llama-3.2-3B-Instruct", "fp16", 10
    ),
    "mixtral_8x7b": ModelSpec(
        "mistralai/Mixtral-8x7B-Instruct-v0.1", "nf4", 26
    ),
}

if (
    len(PANEL) != len(set(PANEL))
    or not set(PANEL).issubset(REGISTRY)
):
    raise ValueError("PANEL must contain unique known keys.")

if (
    len(RUN_MODELS) != len(set(RUN_MODELS))
    or not set(RUN_MODELS).issubset(PANEL)
    or SESSION_MINUTES <= 0
    or CALIBRATION_FOLDS < 2
):
    raise ValueError(
        "Invalid run selection, session budget or fold count."
    )

if CONTEXT_CAP is not None and CONTEXT_CAP < 2:
    raise ValueError(
        "CONTEXT_CAP must be None or at least two tokens."
    )

if RUN_MODELS and not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a Kaggle GPU. Set RUN_MODELS=[] only to analyze "
        "a saved checkpoint."
    )

OUT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False


def stable_hash(value):
    return hashlib.sha256(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        ).encode("utf-8")
    ).hexdigest()


def write_json(path, value):
    Path(path).write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )


def secret(name):
    value = os.environ.get(name, "").strip()

    if not value:
        try:
            from kaggle_secrets import UserSecretsClient

            value = UserSecretsClient().get_secret(name).strip()
        except Exception:
            value = ""

    return value or None


HF_TOKEN = secret("HF_TOKEN")
hub = HfApi(token=HF_TOKEN)

# Validate the checkpoint before copying it into the working folder.
checkpoint_to_read = DB if DB.is_file() else Path(RESUME_DB)

connection = sqlite3.connect(
    checkpoint_to_read.resolve().as_uri() + "?mode=ro",
    uri=True,
)
try:
    saved = connection.execute(
        "SELECT value FROM meta WHERE key='protocol'"
    ).fetchone()
    previous_protocol = json.loads(saved[0]) if saved else None
finally:
    connection.close()

if (
    not isinstance(previous_protocol, dict)
    or previous_protocol.get("version") != "3.0.0"
):
    raise RuntimeError(
        "The selected database is not a compatible v3 checkpoint."
    )

if not previous_protocol.get("dataset_sha"):
    raise RuntimeError(
        "The checkpoint protocol is missing dataset_sha."
    )

if not DB.exists():
    shutil.copy2(checkpoint_to_read, DB)
    print("Copied saved checkpoint to:", DB)

# Resume with the exact dataset revision from the previous run.
DATASET_REVISION = previous_protocol["dataset_sha"]

print("Full-dataset local panel:", PANEL)
print("Selected this session:", RUN_MODELS)
print("Checkpoint:", DB)
print("Dataset revision:", DATASET_REVISION)
print("Previous predictions will be retained.")
print("No hosted inference is configured.")

Checkpoint selected for resume: /kaggle/input/datasets/dipta2002/banglallm-dataset/checkpoint_export.sqlite
Copied saved checkpoint to: /kaggle/working/bnmmlu_v3_full/checkpoint.sqlite
Full-dataset local panel: ['qwen_1_5b', 'smol_1_7b', 'titu_3b', 'tiger_1b', 'llama3_3b', 'mixtral_8x7b']
Selected this session: ['llama3_3b', 'tiger_1b', 'mixtral_8x7b']
Checkpoint: /kaggle/working/bnmmlu_v3_full/checkpoint.sqlite
Dataset revision: 6e4c8d165855cd61387a2f72937336eaadc8212e
Previous predictions will be retained.
No hosted inference is configured.


## Audit every dataset row
Duplicates and conflicting answers are flagged, not discarded. The source manifest covers all source splits. A question-group hash assigns cross-fitting folds without inspecting model outcomes. Invalid schemas receive an explicit reason.

In [3]:
def norm(value):
    return " ".join(unicodedata.normalize("NFC", value).split())

def prepare_rows(raw_splits):
    items, manifest = [], []
    for split, rows in raw_splits.items():
        for index, row in enumerate(rows):
            row_id = f"{split}:{index}"
            source_id = str(row.get("Unique_Serial", ""))
            subject = row.get("subject_name")
            question = row.get("question")
            options = row.get("options")
            if isinstance(options, np.ndarray):
                options = options.tolist()
            answer = str(row.get("correct_answer", "")).strip().upper()
            valid = (isinstance(question, str) and bool(norm(question))
                     and isinstance(subject, str) and bool(norm(subject))
                     and isinstance(options, list) and len(options) == 4
                     and all(isinstance(v, str) and bool(norm(v)) for v in options)
                     and answer in list("ABCD"))
            entry = dict(row_id=row_id, source_split=split, source_id=source_id,
                         valid=bool(valid), reason="" if valid else "invalid_schema_or_value")
            if valid:
                question, options = norm(question), [norm(v) for v in options]
                group = stable_hash(question)
                y = "ABCD".index(answer)
                fold = int(stable_hash(["crossfit", SEED, group])[:16], 16) % CALIBRATION_FOLDS
                item = dict(row_id=row_id, source_id=source_id, subject=norm(subject),
                            question=question, options=options, y=y, question_group=group,
                            fold=fold, content_hash=stable_hash([question, options, y]))
                items.append(item)
                entry.update(subject=item["subject"], question_group=group, fold=fold,
                             content_hash=item["content_hash"], duplicate_options=len(set(options)) != 4,
                             answer_hash=stable_hash(options[y]))
            manifest.append(entry)
    frame = pd.DataFrame(manifest)
    if not items:
        raise RuntimeError("No structurally valid rows; inspect the source schema.")
    valid_frame = frame[frame.valid]
    counts = valid_frame.groupby("question_group").size()
    conflicts = valid_frame.groupby("question_group").answer_hash.nunique().gt(1)
    frame["repeated_question"] = frame.question_group.map(counts).fillna(0).gt(1)
    frame["conflicting_answer_text"] = frame.question_group.map(conflicts).eq(True)
    frame = frame.drop(columns="answer_hash")
    return items, frame

try:
    data_sha = hub.dataset_info(DATASET, revision=DATASET_REVISION).sha
    raw = load_dataset(DATASET, DATASET_CONFIG, revision=data_sha, token=HF_TOKEN)
except Exception:
    raise RuntimeError("Dataset access failed. Enable Internet and the HF_TOKEN secret, and accept BnMMLU access terms.") from None
source_counts = {name: len(split) for name, split in raw.items()}
items, source_manifest = prepare_rows(raw)
del raw
gc.collect()
source_manifest.to_csv(OUT / "source_manifest.csv", index=False)
source_manifest[~source_manifest.valid].to_csv(OUT / "invalid_source_rows.csv", index=False)
write_json(OUT / "data_audit.json", dict(dataset=DATASET, config=DATASET_CONFIG, dataset_sha=data_sha,
    source_split_counts=source_counts, source_rows=len(source_manifest), valid_rows=len(items),
    invalid_rows=int((~source_manifest.valid).sum()), unique_question_groups=source_manifest.question_group.nunique(),
    repeated_question_rows=int(source_manifest.repeated_question.sum()),
    conflicting_answer_rows=int(source_manifest.conflicting_answer_text.sum()),
    sampled=False, deduplicated=False))
print("All source rows:", len(source_manifest), "| valid rows scheduled per model:", len(items))
print("No sampling or development-pool filter applied.")
display(source_manifest.groupby(["source_split", "valid"]).size().rename("rows"))

README.md: 0.00B [00:00, ?B/s]

data/full/bnmmlu-full.parquet:   0%|          | 0.00/15.4M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

All source rows: 134382 | valid rows scheduled per model: 134375
No sampling or development-pool filter applied.


source_split  valid
train         False         7
              True     134375
Name: rows, dtype: int64

## Scoring and calibration helpers
The plain-completion prompt and four prefix-free answer-label events match the earlier experiment. Confidence is conditional on these four label events. All tokens in multi-token labels are scored. The full-dataset NLL is calculated from log probabilities, avoiding clipping of extreme negative log-likelihoods. Prompt style and mixed precision remain limitations when comparing instruction models.

In [4]:
def prompt_text(r, rotation):
    order = np.roll(np.arange(4), -rotation).tolist()
    choices = "\n".join(f"{label}. {r['options'][j]}" for label, j in zip("ABCD", order))
    return (
        "Choose the correct answer. Reply with one letter: A, B, C, or D.\n"
        + "Question: "
        + r["question"]
        + "\n"
        + choices
        + "\nAnswer:"
    ), order


def encoded_prompt(tok, text):
    return tok.encode(text, add_special_tokens=True)


def label_tokens(tok):
    labels = [tok.encode(" " + c, add_special_tokens=False) for c in "ABCD"]
    assert all(labels) and len({tuple(x) for x in labels}) == 4
    for i, a in enumerate(labels):
        for j, b in enumerate(labels):
            if i != j:
                assert a != b[: len(a)], "Non-prefix-free label token sequences."
    return labels


def fragment_features(tok, r):
    text = "\n".join([r["question"], *r["options"]])
    ids = tok.encode(text, add_special_tokens=False)
    words = len(text.split())
    graphemes = len(regex.findall(r"\X", text))
    return dict(
        fertility=len(ids) / max(words, 1),
        tokens_per_grapheme=len(ids) / max(graphemes, 1),
        content_tokens=len(ids),
        words=words,
        characters=len(text),
        mean_option_words=float(np.mean([len(o.split()) for o in r["options"]])),
        unk_rate=sum(t == tok.unk_token_id for t in ids) / max(len(ids), 1),
        has_markup=int("<" in text and ">" in text),
    )


def model_input_device(model):
    return model.get_input_embeddings().weight.device


def trailing_logits(model, token_ids, positions):
    inputs = torch.tensor([token_ids], dtype=torch.long, device=model_input_device(model))
    kwargs = {"input_ids": inputs, "attention_mask": torch.ones_like(inputs), "use_cache": False}
    parameters = inspect.signature(model.forward).parameters
    if "logits_to_keep" in parameters:
        kwargs["logits_to_keep"] = positions
    elif "num_logits_to_keep" in parameters:
        kwargs["num_logits_to_keep"] = positions
    output = model(**kwargs)
    # Cloning the small slice releases the full [sequence, vocabulary] tensor on older models.
    logits = output.logits[0, -positions:, :].float().clone()
    del output, inputs
    return logits


@torch.inference_mode()
def score_labels(model, tokenizer, text, labels):
    prompt_ids = encoded_prompt(tokenizer, text)
    if not prompt_ids:
        raise ValueError("The prompt tokenized to an empty sequence.")
    if all(len(label) == 1 for label in labels):
        logits = trailing_logits(model, prompt_ids, 1)[0]
        log_probabilities = torch.log_softmax(logits, dim=-1)
        scores = log_probabilities[[label[0] for label in labels]].cpu().numpy().astype(float)
    else:
        scores = []
        for label in labels:
            # The last prompt position predicts label[0]; the last supplied label token is not needed.
            logits = trailing_logits(model, prompt_ids + label[:-1], len(label))
            targets = torch.tensor(label, device=logits.device)
            token_log_probabilities = torch.log_softmax(logits, dim=-1)
            value = token_log_probabilities.gather(1, targets[:, None]).sum()
            scores.append(float(value.cpu()))
        scores = np.asarray(scores)
    if not np.isfinite(scores).all():
        raise FloatingPointError(
            "Nonfinite likelihoods. Stop and investigate precision; do not reuse this result."
        )
    mass = float(np.exp(logsumexp(scores)))
    if mass > 1.0001:
        raise ValueError("Candidate events are not disjoint or their probabilities are invalid.")
    return scores, mass, len(prompt_ids)


def metrics(p, y):
    p = np.asarray(p, float)
    y = np.asarray(y, int)
    assert len(p) and p.shape == (len(y), 4)
    assert np.isfinite(p).all() and (p >= 0).all() and np.allclose(p.sum(1), 1)
    correct = p.argmax(1) == y
    conf = p.max(1)
    out = {
        "accuracy": float(correct.mean()),
        "brier": float(((p - np.eye(4)[y]) ** 2).sum(1).mean()),
        "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1)).mean()),
        "overconfidence": float((conf - correct).mean()),
    }
    for bins in [10, 15, 20]:
        ids = np.minimum((conf * bins).astype(int), bins - 1)
        residual_sums = np.bincount(ids, weights=conf - correct, minlength=bins)
        out[f"ece{bins}"] = float(np.abs(residual_sums).sum() / len(y))
    return out


def fit_temperature(scores, y):
    y = np.asarray(y, int)
    s = np.asarray(scores, float)

    def loss(logt):
        z = s / np.exp(logt)
        return float((logsumexp(z, axis=1) - z[np.arange(len(y)), y]).mean())

    opt = minimize_scalar(loss, bounds=(-4, 4), method="bounded")
    assert opt.success
    return float(np.exp(opt.x)), bool(abs(opt.x) > 3.95)


def score_metrics(scores, y, temperatures=1.0):
    scores, y = np.asarray(scores, float), np.asarray(y, int)
    t = np.asarray(temperatures, float)
    if t.ndim == 1:
        t = t[:, None]
    if not np.isfinite(t).all() or not (t > 0).all():
        raise ValueError("Temperature must be finite and positive.")
    z = scores / t
    lp = z - logsumexp(z, axis=1, keepdims=True)
    p = np.exp(lp)
    result = metrics(p, y)
    nll = -lp[np.arange(len(y)), y]
    result["nll"] = float(nll.mean())
    return result, p, nll

def crossfit_temperatures(scores, y, folds, groups):
    scores, y, folds, groups = map(np.asarray, (scores, y, folds, groups))
    if len(set(folds.tolist())) != CALIBRATION_FOLDS:
        raise ValueError("Not all cross-fitting folds have eligible rows.")
    assignments = pd.DataFrame(dict(group=groups, fold=folds)).groupby("group").fold.nunique()
    if assignments.gt(1).any():
        raise ValueError("A question group crosses calibration folds.")
    temperatures = np.empty(len(y))
    audit = []
    for fold in range(CALIBRATION_FOLDS):
        test = folds == fold
        train = ~test
        if len(set(groups[train].tolist())) < 20 or not test.any():
            raise ValueError("Too few independent training groups for cross-fitting.")
        value, boundary = fit_temperature(scores[train], y[train])
        temperatures[test] = value
        audit.append(dict(fold=fold, train_rows=int(train.sum()), test_rows=int(test.sum()),
                          temperature=value, boundary=boundary))
    return temperatures, audit

assert np.isclose(score_metrics(np.zeros((4, 4)), np.arange(4))[0]["brier"], .75)
assert np.isclose(score_metrics(np.zeros((4, 4)), np.arange(4))[0]["nll"], np.log(4))
assert score_metrics(np.array([[0., -1000., -1000., -1000.]]), np.array([1]))[0]["nll"] == 1000.
print("Metric sanity checks passed.")

Metric sanity checks passed.


## Freeze dataset/code identity and open the checkpoint
The global protocol covers data, row order, scoring helpers, precision, context policy and environment. Each model's first successful preparation separately freezes its immutable Hugging Face revision and tokenizer/config fingerprints. A blocked model can be prepared in a later session without changing the rows already scored by other models.

In [5]:
def constant_signature(value):
    if isinstance(value, types.CodeType):
        return code_signature(value)
    if isinstance(value, (set, frozenset)):
        parts = [constant_signature(item) for item in value]
        return {"set": sorted(parts, key=lambda item: json.dumps(item, sort_keys=True))}
    if isinstance(value, tuple):
        return {"tuple": [constant_signature(item) for item in value]}
    if isinstance(value, bytes):
        return {"bytes": value.hex()}
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return {"literal": repr(value)}


def code_signature(code_object):
    return {
        "bytecode": code_object.co_code.hex(),
        "constants": [constant_signature(value) for value in code_object.co_consts],
        "names": code_object.co_names,
        "variables": code_object.co_varnames,
    }


def function_signature(function):
    original = inspect.unwrap(function)
    return code_signature(original.__code__)


import types
versions = {name: installed_version(name) for name in (
    "torch", "transformers", "huggingface_hub", "datasets", "accelerate", "bitsandbytes",
    "numpy", "pandas", "scipy", "statsmodels", "regex", "sentencepiece", "tokenizers")}
protocol = dict(version="3.0.0", dataset=DATASET, config=DATASET_CONFIG, dataset_sha=data_sha,
    panel={key:vars(REGISTRY[key]) for key in PANEL}, seed=SEED, calibration_folds=CALIBRATION_FOLDS,
    context_cap=CONTEXT_CAP, rotations=ROTATIONS, source_rows=len(source_manifest),
    items_hash=stable_hash([(r["row_id"],r["content_hash"],r["fold"]) for r in items]),
    population="all_source_rows_without_deduplication", packages=versions, python=platform.python_version(),
    scoring="plain_completion_prefix_free_answer_labels", calibration="grouped_out_of_fold_temperature",
    functions={f.__name__:function_signature(f) for f in (
        prepare_rows, norm, prompt_text, encoded_prompt, label_tokens, score_labels, trailing_logits,
        fragment_features, score_metrics, crossfit_temperatures, fit_temperature, metrics)})
fingerprint = stable_hash(protocol)
con = sqlite3.connect(DB, timeout=60)
con.execute("CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT)")
con.execute("CREATE TABLE IF NOT EXISTS scores (model TEXT, row_id TEXT, status TEXT, payload TEXT, PRIMARY KEY(model,row_id))")
con.execute("CREATE TABLE IF NOT EXISTS events (timestamp TEXT, model TEXT, event TEXT, detail TEXT)")
saved = con.execute("SELECT value FROM meta WHERE key='fingerprint'").fetchone()
if saved and saved[0] != fingerprint:
    con.close()
    raise RuntimeError("Checkpoint protocol mismatch. Restore its settings/environment or use a new output directory.")
con.execute("INSERT OR REPLACE INTO meta VALUES ('protocol',?)", (json.dumps(protocol),))
con.execute("INSERT OR REPLACE INTO meta VALUES ('fingerprint',?)", (fingerprint,))
con.commit()
write_json(OUT / "manifest.json", dict(protocol, fingerprint=fingerprint))

def log_event(model, event, detail):
    con.execute("INSERT INTO events VALUES (?,?,?,?)", (datetime.now(timezone.utc).isoformat(), model, event, str(detail)))
    con.commit()

def save_checkpoint_snapshot():
    con.commit()
    temporary = OUT / "checkpoint_export.tmp.sqlite"
    backup = sqlite3.connect(temporary)
    try:
        con.backup(backup)
    finally:
        backup.close()
    temporary.replace(OUT / "checkpoint_export.sqlite")

def safe_error(exc):
    text = f"{type(exc).__name__}: {str(exc)}"
    if HF_TOKEN:
        text = text.replace(HF_TOKEN, "[REDACTED]")
    return text[:1000]

def progress_table():
    rows = []
    for key in PANEL:
        repo = REGISTRY[key].repo
        counts = dict(con.execute("SELECT status, COUNT(*) FROM scores WHERE model=? GROUP BY status", (repo,)))
        scored, context, errors = [counts.get(s, 0) for s in ("ok", "context_exceeded", "error")]
        pending = len(items) - scored - context - errors
        event = con.execute("SELECT event,detail FROM events WHERE model=? ORDER BY rowid DESC LIMIT 1", (repo,)).fetchone()
        status = ("complete" if scored == len(items) else "processed_with_exclusions"
                  if pending == 0 and errors == 0 else "not_selected" if key not in RUN_MODELS
                  else event[0] if event else "pending")
        rows.append(dict(model=repo, precision=REGISTRY[key].precision, status=status,
            source_rows=len(source_manifest), valid_rows=len(items), scored_rows=scored,
            invalid_source_rows=len(source_manifest)-len(items), context_exceeded_rows=context,
            failed_rows=errors, pending_rows=pending, coverage_source=scored/len(source_manifest),
            processing_complete=pending == 0 and errors == 0,
            all_source_rows_scored=scored == len(source_manifest), detail=event[1] if event else ""))
    return pd.DataFrame(rows)

## Local model preparation and resource checks
Small models load directly onto a GPU. Mixtral uses explicit NF4 double quantization and FP16 compute, with GPU sharding and no CPU/disk offload. Disk space is checked against the official weight files before download. A gated-model or resource failure is recorded and does not stop the other models. The shared source population stays fixed; each model reports its own context coverage.

In [6]:
class ResourceBlocked(RuntimeError):
    pass

def cache_directory(repo):
    path = CACHE_ROOT / repo.replace("/", "__")
    path.mkdir(parents=True, exist_ok=True)
    return path

def prepare_model(key):
    spec = REGISTRY[key]
    saved = con.execute("SELECT value FROM meta WHERE key=?", ("model::" + spec.repo,)).fetchone()
    previous = json.loads(saved[0]) if saved else None
    revision = previous["revision"] if previous else "main"
    info = hub.model_info(spec.repo, revision=revision, files_metadata=True)
    config = AutoConfig.from_pretrained(spec.repo, revision=info.sha, token=HF_TOKEN,
                                      cache_dir=cache_directory(spec.repo), trust_remote_code=False)
    tok = AutoTokenizer.from_pretrained(spec.repo, revision=info.sha, token=HF_TOKEN,
                                       cache_dir=cache_directory(spec.repo), trust_remote_code=False)
    labels = label_tokens(tok)
    text_config = getattr(config, "text_config", config)
    limits = [getattr(text_config, "max_position_embeddings", None), getattr(tok, "model_max_length", None), CONTEXT_CAP]
    limits = [int(v) for v in limits if v is not None and 1 < v < 10**8]
    if not limits:
        raise ValueError("Cannot establish the model context limit from its configuration.")
    limit = min(limits)
    weights = [f for f in info.siblings if re.fullmatch(r"model(?:-\d+-of-\d+)?\.safetensors", f.rfilename)]
    if not weights or any(f.size is None for f in weights):
        raise ValueError("Cannot determine the official safetensors download size; no guessed size is used.")
    cfg = config.to_dict()
    cfg.pop("_name_or_path", None)
    cfg.pop("_commit_hash", None)
    identity = dict(repo=spec.repo, revision=info.sha, precision=spec.precision,
                    config_hash=stable_hash(cfg), tokenizer_class=type(tok).__name__,
                    vocabulary_hash=stable_hash(tok.get_vocab()),
                    backend_hash=stable_hash(tok.backend_tokenizer.to_str()) if tok.is_fast else None,
                    label_ids=labels, context_limit=limit,
                    weights={f.rfilename:f.size for f in weights})
    if previous and previous != identity:
        raise RuntimeError("Frozen model/tokenizer identity changed; refusing mixed results.")
    con.execute("INSERT OR REPLACE INTO meta VALUES (?,?)", ("model::"+spec.repo, json.dumps(identity)))
    con.commit()
    write_json(OUT / (key + "_model_identity.json"), identity)
    write_json(OUT / (key + "_config.json"), config.to_dict())
    return tok, labels, identity

def load_model(key, identity):
    spec = REGISTRY[key]
    free = {i:torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())}
    if not free:
        raise ResourceBlocked("No CUDA GPU is available.")
    gpu = max(free, key=free.get)
    available = sum(free.values()) if spec.precision == "nf4" else free[gpu]
    if available < spec.min_free_gpu_gib * 2**30:
        raise ResourceBlocked(f"{key}: need at least {spec.min_free_gpu_gib:g} GiB free GPU memory; available={available/2**30:.1f} GiB. Mixtral requires multiple sufficiently large GPUs.")
    cache = cache_directory(spec.repo)
    snapshot = cache / ("models--" + spec.repo.replace("/", "--")) / "snapshots" / identity["revision"]
    remaining = sum(size for name,size in identity["weights"].items() if not (snapshot / name).is_file())
    if shutil.disk_usage(cache).free < remaining + 2 * 2**30:
        raise ResourceBlocked(f"{key}: need {remaining/2**30:.1f} GiB of weight-download space plus 2 GiB reserve in {cache}. NF4 still downloads the official full weights.")
    kwargs = dict(revision=identity["revision"], token=HF_TOKEN, cache_dir=cache,
                  dtype=torch.float32 if spec.precision=="fp32" else torch.float16,
                  low_cpu_mem_usage=True, trust_remote_code=False)
    if spec.precision == "nf4":
        kwargs.update(quantization_config=BitsAndBytesConfig(load_in_4bit=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
            device_map="auto", max_memory={i:int(v-1.5*2**30) for i,v in free.items() if v>3*2**30})
    else:
        kwargs["device_map"] = {"":gpu}
    model = AutoModelForCausalLM.from_pretrained(spec.repo, **kwargs)
    if any(str(v) in {"cpu","disk"} for v in getattr(model,"hf_device_map",{}).values()):
        del model
        gc.collect()
        torch.cuda.empty_cache()
        raise ResourceBlocked("Loader requires CPU/disk offload. This protocol uses GPU inference only; use more GPU memory.")
    model.eval()
    model.requires_grad_(False)
    return model

def release_cache(repo):
    folder = CACHE_ROOT / repo.replace("/", "__")
    if folder.parent.resolve() != CACHE_ROOT.resolve():
        raise RuntimeError("Refusing cache deletion outside the managed directory.")
    if RELEASE_COMPLETED_WEIGHTS and folder.exists():
        shutil.rmtree(folder)

# Loader code is fingerprinted separately because it is defined after the global protocol.
loader_signature = stable_hash({f.__name__:function_signature(f) for f in (prepare_model, load_model)})
prior = con.execute("SELECT value FROM meta WHERE key='loader_signature'").fetchone()
if prior and prior[0] != loader_signature:
    raise RuntimeError("Loader code changed. Use the original loader or a new output directory.")
con.execute("INSERT OR REPLACE INTO meta VALUES ('loader_signature',?)", (loader_signature,))
con.commit()

## Run all pending rows, model by model
Each result commits immediately. Snapshots are made every 5,000 new records and after each model. Download/permission failures and hardware blocks are model-level events; individual context/forward failures are saved with row IDs. Three consecutive forward errors stop that model to avoid wasting the session. Failed rows can be retried on resume.

In [7]:
deadline = time.monotonic() + SESSION_MINUTES * 60
for key in RUN_MODELS:
    repo = REGISTRY[key].repo
    status_by_row = dict(con.execute("SELECT row_id,status FROM scores WHERE model=?", (repo,)))
    pending = [r for r in items if r["row_id"] not in status_by_row or
               (RETRY_FAILED_ROWS and status_by_row[r["row_id"]] == "error")]
    if not pending:
        print(key, "has no pending rows; saved results will be analyzed.")
        continue
    if time.monotonic() >= deadline:
        log_event(repo, "time_budget", "Deferred before loading; resume this checkpoint.")
        continue
    model = tok = None
    try:
        print(f"Preparing {key}: {len(pending):,} pending of {len(items):,} valid source rows")
        tok, labels, identity = prepare_model(key)
        model = load_model(key, identity)
        score_labels(model, tok, "Choose A. Answer:", labels)
        log_event(repo, "running", json.dumps(dict(gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            cuda=torch.version.cuda, revision=identity["revision"], precision=REGISTRY[key].precision)))
        started, consecutive_errors = time.monotonic(), 0
        for number, item in enumerate(pending, 1):
            if time.monotonic() >= deadline:
                log_event(repo, "time_budget", "Paused between rows; resume to continue.")
                break
            record = {name:item[name] for name in ("row_id","source_id","subject","y","question_group","fold","content_hash")}
            record.update(model=repo, precision=REGISTRY[key].precision)
            try:
                prompt, _ = prompt_text(item, 0)
                tokens = len(encoded_prompt(tok, prompt)) + max(map(len, labels))
                record["required_tokens"] = tokens
                if tokens > identity["context_limit"]:
                    status = "context_exceeded"
                    record["reason"] = f"Requires {tokens} tokens; frozen context limit={identity['context_limit']}. No truncation."
                else:
                    scores, mass, _ = score_labels(model, tok, prompt, labels)
                    record.update(scores=scores.tolist(), candidate_mass=mass, **fragment_features(tok,item))
                    status = "ok"
                consecutive_errors = 0
            except (RuntimeError, ValueError, FloatingPointError) as exc:
                status = "error"
                record["reason"] = safe_error(exc)
                consecutive_errors += 1
                gc.collect()
                torch.cuda.empty_cache()
            con.execute("INSERT OR REPLACE INTO scores VALUES (?,?,?,?)", (repo,item["row_id"],status,json.dumps(record)))
            con.commit()
            if number % 500 == 0 or number == len(pending):
                elapsed = max(time.monotonic()-started,1e-9)
                print(f"{key}: {number:,}/{len(pending):,}; {number/elapsed:.2f} rows/s; estimated remaining {(len(pending)-number)*elapsed/number/3600:.2f} hours")
            if number % 5000 == 0:
                save_checkpoint_snapshot()
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                log_event(repo, "failed", "Consecutive forward errors; inspect row_status.csv after export and retry after fixing the cause.")
                break
        else:
            log_event(repo, "processed", "All pending rows attempted; check per-row coverage and errors.")
    except ResourceBlocked as exc:
        log_event(repo, "resource_blocked", safe_error(exc))
        print(key, "resource blocked:", safe_error(exc))
    except (RuntimeError, OSError, ValueError, ImportError) as exc:
        log_event(repo, "failed", safe_error(exc))
        print(key, "failed:", safe_error(exc))
    finally:
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()
        save_checkpoint_snapshot()
        table = progress_table()
        table.to_csv(OUT / "model_progress.csv", index=False)
    state = table[table.model == repo].iloc[0]
    if state.processing_complete:
        try:
            release_cache(repo)
        except OSError:
            warnings.warn(f"Could not remove the managed cache for {key}; completed scores remain saved.")
display(progress_table())

Preparing llama3_3b: 134,375 pending of 134,375 valid source rows


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

mixtral_8x7b resource blocked: ResourceBlocked: mixtral_8x7b: need at least 26 GiB free GPU memory; available=19.2 GiB. Mixtral requires multiple sufficiently large GPUs.


,model,precision,status,source_rows,valid_rows,scored_rows,invalid_source_rows,context_exceeded_rows,failed_rows,pending_rows,coverage_source,processing_complete,all_source_rows_scored,detail
0,Qwen/Qwen2.5-1.5B-Instruct,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
1,HuggingFaceTB/SmolLM2-1.7B-Instruct,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
2,hishab/titulm-llama-3.2-3b-v2.0,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
3,md-nishat-008/TigerLLM-1B-it,fp32,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
4,meta-llama/Llama-3.2-3B-Instruct,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
5,mistralai/Mixtral-8x7B-Instruct-v0.1,nf4,resource_blocked,134382,134375,0,7,0,0,134375,0.000000,False,False,ResourceBlocked: mixtral_8x7b: need at least 2...


## Full-dataset results and cross-fitted calibration
Raw results are exported even for partial runs, with their scope explicitly marked. Cross-fitted calibration runs only after a model has processed all valid source rows without unresolved errors; context exclusions remain visible. Each row's calibrated prediction is out-of-fold, but this remains a descriptive full-dataset analysis. Per-subject, macro-average and unique-question sensitivity results are separate. Models with different coverage should be compared on common scored rows, not by mixing their separate denominators.

In [8]:
progress = progress_table()
summary_rows, subject_rows, temperature_rows, unique_rows = [], [], [], []
completed = progress.loc[progress.processing_complete & progress.scored_rows.gt(0), "model"].tolist()
common_ids = None
for repo in completed:
    ids = {r[0] for r in con.execute("SELECT row_id FROM scores WHERE model=? AND status='ok'", (repo,))}
    common_ids = ids if common_ids is None else common_ids & ids
common_rows = []

for key in PANEL:
    repo = REGISTRY[key].repo
    rows = [json.loads(r[0]) for r in con.execute("SELECT payload FROM scores WHERE model=? AND status='ok' ORDER BY row_id", (repo,))]
    if not rows:
        continue
    frame = pd.DataFrame(rows)
    scores, y = np.asarray(frame.scores.tolist()), frame.y.to_numpy()
    state = progress[progress.model == repo].iloc[0]
    scope = "all_valid_rows" if len(frame)==len(items) else "context_eligible_rows" if state.processing_complete else "partial_run"
    methods = [("raw", np.ones(len(frame)))]
    if state.processing_complete:
        try:
            temperatures, audit = crossfit_temperatures(scores,y,frame.fold.to_numpy(),frame.question_group.to_numpy())
            methods.append(("crossfit_temperature",temperatures))
            temperature_rows.extend(dict(model=repo,**r) for r in audit)
        except (ValueError, AssertionError) as exc:
            log_event(repo,"calibration_unavailable",safe_error(exc))
            print(key,"calibration unavailable:",safe_error(exc))
    exported = frame.drop(columns=["scores"]).copy()
    for method, temperatures in methods:
        result,p,nll = score_metrics(scores,y,temperatures)
        summary_rows.append(dict(model=repo,precision=REGISTRY[key].precision,method=method,scope=scope,
            source_rows=len(source_manifest),valid_rows=len(items),scored_rows=len(frame),
            coverage_source=len(frame)/len(source_manifest),all_source_rows_scored=bool(state.all_source_rows_scored),**result))
        exported[method+"_prediction"] = p.argmax(1)
        exported[method+"_confidence"] = p.max(1)
        exported[method+"_correct"] = (p.argmax(1)==y).astype(int)
        exported[method+"_nll"] = nll
        for label_index,label in enumerate("ABCD"):
            exported[method+"_p_"+label] = p[:,label_index]
        for subject, indices in frame.groupby("subject").indices.items():
            values = metrics(p[indices],y[indices])
            values["nll"] = float(nll[indices].mean())
            subject_rows.append(dict(model=repo,method=method,subject=subject,rows=len(indices),scope=scope,**values))
        unique_indices = np.flatnonzero(~frame.question_group.duplicated().to_numpy())
        values = metrics(p[unique_indices],y[unique_indices])
        values["nll"] = float(nll[unique_indices].mean())
        unique_rows.append(dict(model=repo,method=method,scope=scope,question_groups=len(unique_indices),**values))
        if method=="raw" and repo in completed and common_ids and len(completed)>1:
            mask = frame.row_id.isin(common_ids).to_numpy()
            values = metrics(p[mask],y[mask])
            values["nll"] = float(nll[mask].mean())
            common_rows.append(dict(model=repo,common_rows=int(mask.sum()),models_in_comparison=len(completed),
                entire_panel_complete=len(completed)==len(PANEL),**values))
    exported.to_csv(OUT / (key+"_row_predictions.csv"),index=False)
    del frame, rows, scores, exported
    gc.collect()

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "metrics_full_dataset.csv",index=False)
subjects = pd.DataFrame(subject_rows)
subjects.to_csv(OUT / "subject_metrics.csv",index=False)
if not subjects.empty:
    subjects.groupby(["model","method","scope"])[["accuracy","brier","nll","ece15"]].mean().reset_index().to_csv(OUT / "subject_macro_metrics.csv",index=False)
pd.DataFrame(temperature_rows).to_csv(OUT / "crossfit_temperatures.csv",index=False)
pd.DataFrame(unique_rows).to_csv(OUT / "unique_question_sensitivity.csv",index=False)
pd.DataFrame(common_rows).to_csv(OUT / "common_rows_raw_comparison.csv",index=False)
write_json(OUT / "common_rows_comparison_scope.json",dict(models=completed,common_rows=len(common_ids or []),
    entire_panel_complete=len(completed)==len(PANEL),descriptive_only=True))
if not summary.empty:
    display(summary[["model","method","scope","scored_rows","coverage_source","accuracy","brier","nll","ece15"]])
else:
    print("No successful predictions yet. Inspect model_progress.csv and run_events.csv; the checkpoint is still exportable.")

,model,method,scope,scored_rows,coverage_source,accuracy,brier,nll,ece15
0,Qwen/Qwen2.5-1.5B-Instruct,raw,all_valid_rows,134375,0.999948,0.316904,0.890331,1.761462,0.300181
1,Qwen/Qwen2.5-1.5B-Instruct,crossfit_temperature,all_valid_rows,134375,0.999948,0.316904,0.736940,1.362751,0.016823
2,HuggingFaceTB/SmolLM2-1.7B-Instruct,raw,all_valid_rows,134375,0.999948,0.282389,0.938719,1.854514,0.327940
3,HuggingFaceTB/SmolLM2-1.7B-Instruct,crossfit_temperature,all_valid_rows,134375,0.999948,0.282389,0.746182,1.378801,0.009912
4,hishab/titulm-llama-3.2-3b-v2.0,raw,all_valid_rows,134375,0.999948,0.266530,0.866417,1.681904,0.242593
5,hishab/titulm-llama-3.2-3b-v2.0,crossfit_temperature,all_valid_rows,134375,0.999948,0.266530,0.749300,1.384919,0.001051
6,md-nishat-008/TigerLLM-1B-it,raw,all_valid_rows,134375,0.999948,0.322486,0.755941,1.402805,0.110756
7,md-nishat-008/TigerLLM-1B-it,crossfit_temperature,all_valid_rows,134375,0.999948,0.322486,0.734672,1.358133,0.020256
8,meta-llama/Llama-3.2-3B-Instruct,raw,all_valid_rows,134375,0.999948,0.386865,0.708907,1.318543,0.093768
9,meta-llama/Llama-3.2-3B-Instruct,crossfit_temperature,all_valid_rows,134375,0.999948,0.386865,0.693365,1.283251,0.018384


## Export and resume
The archive excludes model weights and source question/option text. It includes the full source-row audit, SQLite checkpoint, per-model predictions, metrics, fold temperatures and failure details. Keep it private unless sharing complies with dataset/model terms. A zero-score or blocked model has no fabricated metrics row; it remains visible in the coverage table.

In [9]:
import csv
progress = progress_table()
progress.to_csv(OUT / "model_progress.csv",index=False)
pd.read_sql_query("SELECT * FROM events",con).to_csv(OUT / "run_events.csv",index=False)
with (OUT / "row_status.csv").open("w",encoding="utf-8",newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["model","row_id","status","reason"])
    for model,row_id,status,payload in con.execute("SELECT model,row_id,status,payload FROM scores ORDER BY model,row_id"):
        writer.writerow([model,row_id,status,json.loads(payload).get("reason","")])
write_json(OUT / "run_status.json",dict(version="3.0.0",fingerprint=fingerprint,
    source_rows=len(source_manifest),valid_rows=len(items),selected_this_session=RUN_MODELS,
    all_models_processed=bool(progress.processing_complete.all()),
    all_models_scored_every_source_row=bool(progress.all_source_rows_scored.all()),
    inference_location="local_Kaggle",descriptive_full_dataset=True,publication_ready_claim=False,
    model_progress=progress.to_dict("records")))
save_checkpoint_snapshot()
con.close()
archive = shutil.make_archive(str(OUT.parent / (OUT.name+"_results")),"zip",OUT)
display(progress)
display(FileLink(archive))
print("Saved:",archive)
print("Resume a v3 run using checkpoint_export.sqlite, the same settings, and any desired RUN_MODELS subset.")

,model,precision,status,source_rows,valid_rows,scored_rows,invalid_source_rows,context_exceeded_rows,failed_rows,pending_rows,coverage_source,processing_complete,all_source_rows_scored,detail
0,Qwen/Qwen2.5-1.5B-Instruct,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
1,HuggingFaceTB/SmolLM2-1.7B-Instruct,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
2,hishab/titulm-llama-3.2-3b-v2.0,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
3,md-nishat-008/TigerLLM-1B-it,fp32,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
4,meta-llama/Llama-3.2-3B-Instruct,fp16,complete,134382,134375,134375,7,0,0,0,0.999948,True,False,All pending rows attempted; check per-row cove...
5,mistralai/Mixtral-8x7B-Instruct-v0.1,nf4,resource_blocked,134382,134375,0,7,0,0,134375,0.000000,False,False,ResourceBlocked: mixtral_8x7b: need at least 2...


/kaggle/working/bnmmlu_v3_full_results.zip

Saved: /kaggle/working/bnmmlu_v3_full_results.zip
Resume a v3 run using checkpoint_export.sqlite, the same settings, and any desired RUN_MODELS subset.


## Interpretation and validation
Full-dataset raw metrics describe model performance on the scored source rows. Cross-fitted temperature scaling changes confidence, not answer ordering; unchanged accuracy is expected. Duplicate and conflicting items remain in source-weighted results; inspect the source audit and the unique-question sensitivity. Confidence is conditional on answer-label events, and low candidate mass can make these probabilities misleading as unconditional confidence.

The earlier four-model smoke run succeeded, but it does not validate new Llama/Mixtral downloads, NF4 sharding or runtime on the entire dataset. See the accompanying validation report for tests actually run. No API-model code is retained. Local model loading can still be blocked by account permissions, changing environments, disk limits or GPU capacity; the report distinguishes these from model answers.

For journal work, preregister follow-up hypotheses, audit contamination and ambiguous questions, justify the comparison panel, and report the Mixtral NF4/Tiger FP32 precision differences. The plain prompt is a controlled format rather than each model's native chat template. This notebook does not treat tokenizer families as independent causal interventions and does not claim Q1 acceptance.